# Speech Denoising — Notebook 06: Evaluation (PESQ)

Objective evaluation of the trained model using **PESQ** (Perceptual Evaluation of Speech Quality) — the industry-standard metric for speech quality, used in ITU-T P.862.

PESQ score range: -0.5 (worst) → 4.5 (best). Scores above 3.5 are considered good quality.

**Methodology:** Run inference on all test chunks, compute PESQ against clean reference, compare against noisy baseline (no model).

## 1. Imports

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import librosa as lb
import matplotlib.pyplot as plt
import soundfile as sf
from pathlib import Path
from IPython.display import Audio
from tqdm import tqdm
from pesq import pesq

## 2. Load Test Data

Test chunks preprocessed in notebook 01 — 1-second segments (16,000 samples) not seen during training.

In [ ]:
test_noisy_chunks = np.load('data/test_noisy_chunks.npy')
test_clean_chunks = np.load('data/test_clean_chunks.npy')

## 3. Model Architecture & Load Weights

In [ ]:
class DenoisingModelDilated(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.LeakyReLU()
        self.layer1 = nn.Conv1d(1, 16, kernel_size=3, dilation=1, padding=1)
        self.layer2 = nn.Conv1d(16, 32, kernel_size=3, dilation=2, padding=2)
        self.layer3 = nn.Conv1d(32, 16, kernel_size=3, dilation=4, padding=4)
        self.layer4 = nn.Conv1d(16, 1, kernel_size=3, dilation=8, padding=8)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        x = self.relu(x)
        x = self.layer4(x)
        x = x.squeeze(1)
        return x

In [ ]:
model = DenoisingModelDilated()
model.load_state_dict(torch.load('denoising_cnn_spectral_loss_4layers_4epochs.pth'))

In [ ]:
model.eval()

## 4. Single Chunk Sanity Check

Verify inference pipeline on one chunk before running the full evaluation loop.

In [ ]:
chunk_tensor = torch.from_numpy(test_noisy_chunks[0]).unsqueeze(0)
pred_clean_chunk = model(chunk_tensor).detach().numpy().squeeze(0)

In [ ]:
score = pesq(16000, test_clean_chunks[0], pred_clean_chunk, 'wb')
print(score)


In [ ]:
score_noisy = pesq(16000, test_clean_chunks[0], test_noisy_chunks[0], 'wb')
print(score_noisy)

## 5. Full Evaluation — Model vs Noisy Baseline

Loop over all test chunks. Track `valid_indices` — chunks where PESQ detects speech. Use same indices for both model and baseline to ensure fair comparison.

`NoUtterancesError` is caught and skipped (silence/noise-only chunks with no detectable speech).

In [ ]:
score_list = []
valid_indices = []
for i, chunk in enumerate(test_noisy_chunks):
    chunk_tensor = torch.from_numpy(chunk).unsqueeze(0)
    pred_clean_chunk = model(chunk_tensor).detach().numpy().squeeze(0)
    try:
        score = pesq(16000, test_clean_chunks[i], pred_clean_chunk, 'wb')
        score_list.append(score)
        valid_indices.append(i)
    except Exception:
        pass

In [ ]:
print(f"Mean PESQ: {np.mean(score_list):.4f}")
print(f"Chunks scored: {len(score_list)}")

In [ ]:
noisy_score_list = []
for i in valid_indices:
    chunk = test_noisy_chunks[i]
    try:
        score = pesq(16000, test_clean_chunks[i], chunk, 'wb')
        noisy_score_list.append(score)
    except Exception:
        pass

In [ ]:
print(f"Mean PESQ: {np.mean(noisy_score_list):.4f}")
print(f"Chunks scored: {len(noisy_score_list)}")

## 6. Results Summary

| | Mean PESQ | Chunks |
|---|---|---|
| Noisy input (no model) | 2.10 | 1643 |
| Model output | 2.04 | 1643 |

**Observation:** Model slightly underperforms the noisy baseline on PESQ. This is expected for a shallow dilated CNN trained for only 4 epochs. PESQ measures mathematical signal similarity, not perceptual quality — informal listening tests suggest the model does reduce some noise components.

**Next steps:** deeper architecture (U-Net), more epochs, or pre-trained model (DeepFilterNet).